# A simple demo of MLProfiler

In [1]:
import json
from pathlib import Path

import httpx

### Utils functions

In [2]:
def load_taxonomy(taxonomy_name: str):
    with open(
            f"resources/taxonomies/{taxonomy_name}_taxonomy.json"
    ) as f:
        return json.load(f)

In [3]:
def _cell_source_as_list(source: list[str] | str) -> list[str]:
    return source if isinstance(source, list) else [source]


def read_notebook_content(notebook_path: Path) -> str:
    with open(notebook_path) as f:
        notebook_content = json.load(f)

    all_python_code: list[str] = [
        block
        for cell in notebook_content["cells"]
        if cell["cell_type"] == "code"
        for block in _cell_source_as_list(cell["source"])
    ]
    return "\n".join(all_python_code)

In [4]:
filename = "nb_11007.ipynb"

python_content = read_notebook_content(Path("resources/notebooks") / filename)
python_content

'import tensorflow as tf\n\nimport math\nfrom tensorflow.examples.tutorials.mnist import input_data\n\nmnist = input_data.read_data_sets("MNIST_data/", one_hot=True, reshape=False, validation_size=0)\nx=mnist.train.images\n\ny=mnist.train.labels\nmnist.test.images.shape\nprint (x.shape)\n\nprint (y.shape)\nX = tf.placeholder("float", [None,28, 28, 1])\n\nY_=tf.placeholder("float",[None,10])\n# variable learning rate\n\nlr = tf.placeholder(tf.float32)\n\n# Probability of keeping a node during dropout = 1.0 at test time (no dropout) and 0.75 at training time\n\npkeep = tf.placeholder(tf.float32)\n# three convolutional layers with their channel counts, and a\n\n# fully connected layer (tha last layer has 10 softmax neurons)\n\nK = 6  # first convolutional layer output depth\n\nL = 12  # second convolutional layer output depth\n\nM = 24  # third convolutional layer\n\nN = 200  # fully connected layer\nW1 = tf.Variable(tf.truncated_normal([5, 5, 1, K], stddev=0.1))  # 5x5 patch, 1 input cha

### Classifying using dspipelines parser, dspipelines profiler and dspipelines taxonomy

In [6]:
request = httpx.post(
    "http://localhost:8081/profile",
    json={
        "notebook_file_stem": filename,
        "python_content": python_content,
        "taxonomy": load_taxonomy("dspipelines"),
        "parser_name": "dspipelines",
        "profiler_name": "dspipelines",
    },
)
request.json()

{'name': 'nb_11007.ipynb',
 'metadata': {'version': '0.4.0-MLProfile',
  'generation_date': '2026-01-30T13:55:48.151150',
  'taxonomy': 'dspipelines',
  'profiler': 'dspipelines',
  'parser': 'dspipelines'},
 'source': [{'name': 'Data Acquisition',
   'tasks': [{'id': '3a10d5e8-d356-4896-b54b-db17cfbf52eb',
     'algoFamily': None,
     'algoName': None,
     'library': '',
     'function': 'read_data_sets',
     'tasks': [{'name': "input_data.read_data_sets('MNIST_data/', one_hot=True, reshape=False, validation_size=0)",
       'tasks': []}],
     'metadata': {'perplexity': 1,
      'logprobs': [[['Data Acquisition', 1.0]]]}}],
   'outputs_ids': []},
  {'name': 'Others',
   'tasks': [{'id': '08eb1f88-9998-4372-b1a4-9858401d09c7',
     'algoFamily': None,
     'algoName': None,
     'library': '',
     'function': 'print',
     'tasks': [{'name': 'print(x.shape)', 'tasks': []}],
     'metadata': {'perplexity': 1, 'logprobs': [[['Others', 1.0]]]}},
    {'id': '20a45616-8abf-4dcf-9602-c0

### Classifying using dspipelines parser, llm profiler and dspipelines taxonomy

In [24]:
request = httpx.post(
    "http://localhost:8081/profile",
    json={
        "notebook_file_stem": filename,
        "context": python_content,
        "python_content": "y = np.concatenate(y)",
        "taxonomy": load_taxonomy("dspipelines"),
        "parser_name": "dspipelines",
        "profiler_name": "llm",
    },
    timeout=20
)
request.json()

{'name': 'nb_11007.ipynb',
 'metadata': {'version': '0.4.0-MLProfile',
  'generation_date': '2026-01-30T14:53:49.120486',
  'taxonomy': 'dspipelines',
  'profiler': 'llm',
  'parser': 'dspipelines'},
 'source': [{'name': 'Others',
   'tasks': [{'id': '80d3fc4b-ceaf-468e-9894-21444ec6df58',
     'algoFamily': None,
     'algoName': None,
     'library': '',
     'function': 'concatenate',
     'tasks': [{'name': 'np.concatenate(y)', 'tasks': []}],
     'metadata': {'perplexity': 1.618870482405536,
      'logprobs': [[['Others', 61.77],
        ['Data', 29.64],
        ['Evaluation', 3.6],
        ['Model', 2.76],
        ['Training', 1.23],
        ['E', 0.55],
        ['Other', 0.28]]]}}],
   'outputs_ids': []}],
 'outputs': {}}